# 05 — Reproducibility and split audit

Audits all completed OOF and locked-test result bundles for:
- frozen split hash consistency;
- expected patient counts;
- identical OOF patient/fold assignments;
- identical locked-test patient set;
- finite primary Dice metrics;
- metadata indicating that ablations/OOF did not evaluate the locked test.

This notebook is useful for the supplementary/reproducibility record.

In [1]:

from pathlib import Path
import io
import json
import math
import shutil
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
EXPECTED_SPLIT_SHA256 = "d5b26261c93305d0554dae92454cee9363c8df5d946c944d47ab1b8a25898459"

print("Python analysis environment ready.")


Python analysis environment ready.


In [2]:

def _json_load_bytes(raw):
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    return json.loads(raw)


def _bundle_candidates(metadata_filename, expected_model):
    """
    Finds a result bundle either as:
    1) a ZIP somewhere under /kaggle/input, or
    2) an already-extracted Kaggle dataset directory.
    """
    candidates = []

    # ZIP bundles.
    for zpath in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zpath, "r") as zf:
                names = zf.namelist()
                if metadata_filename not in names:
                    continue
                meta = _json_load_bytes(zf.read(metadata_filename))
                if str(meta.get("model", "")) == expected_model:
                    candidates.append({
                        "kind": "zip",
                        "source": zpath,
                        "metadata": meta,
                        "metadata_filename": metadata_filename,
                    })
        except zipfile.BadZipFile:
            continue

    # Extracted bundles.
    for mpath in INPUT_ROOT.rglob(metadata_filename):
        try:
            meta = json.loads(mpath.read_text(encoding="utf-8"))
        except Exception:
            continue
        if str(meta.get("model", "")) == expected_model:
            candidates.append({
                "kind": "dir",
                "source": mpath.parent,
                "metadata": meta,
                "metadata_filename": metadata_filename,
            })

    if not candidates:
        raise FileNotFoundError(
            f"Could not find result bundle for model={expected_model!r}. "
            f"Attach the corresponding result ZIP to Kaggle."
        )

    # If the same bundle is attached more than once, choose the first
    # deterministic candidate and print a warning.
    candidates = sorted(candidates, key=lambda x: str(x["source"]))
    if len(candidates) > 1:
        print(
            f"Warning: found {len(candidates)} matching bundles for "
            f"{expected_model}; using {candidates[0]['source']}"
        )

    bundle = candidates[0]

    split_hash = bundle["metadata"].get("split_sha256")
    if split_hash is not None and split_hash != EXPECTED_SPLIT_SHA256:
        raise RuntimeError(
            f"Frozen split hash mismatch for {expected_model}: {split_hash}"
        )

    return bundle


def _read_csv_from_bundle(bundle, required_suffix):
    if bundle["kind"] == "zip":
        with zipfile.ZipFile(bundle["source"], "r") as zf:
            hits = [
                name for name in zf.namelist()
                if name.endswith(required_suffix)
            ]
            if len(hits) != 1:
                raise FileNotFoundError(
                    f"Expected exactly one *{required_suffix} inside "
                    f"{bundle['source']}, found {hits}"
                )
            return pd.read_csv(zf.open(hits[0]))

    hits = list(bundle["source"].rglob(f"*{required_suffix}"))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"Expected exactly one *{required_suffix} under "
            f"{bundle['source']}, found {hits}"
        )
    return pd.read_csv(hits[0])


def _read_optional_csv_from_bundle(bundle, required_suffix):
    try:
        return _read_csv_from_bundle(bundle, required_suffix)
    except Exception:
        return None


def load_oof_model(expected_model):
    bundle = _bundle_candidates(
        "evaluation_metadata.json",
        expected_model,
    )
    df = _read_csv_from_bundle(bundle, "patient_metrics.csv").copy()

    if len(df) != 80 or df["patient_id"].nunique() != 80:
        raise RuntimeError(
            f"{expected_model}: expected 80 unique OOF patients, "
            f"found rows={len(df)}, unique={df['patient_id'].nunique()}."
        )

    if "fold" not in df.columns:
        raise RuntimeError(f"{expected_model}: OOF patient file has no fold column.")

    if bundle["metadata"].get("locked_test_evaluated") not in (False, None):
        raise RuntimeError(
            f"{expected_model}: OOF metadata says locked test was evaluated."
        )

    df["patient_id"] = df["patient_id"].astype(int)
    df = df.sort_values("patient_id").reset_index(drop=True)
    return bundle, df


def load_test_model(expected_model):
    bundle = _bundle_candidates(
        "final_locked_test_metadata.json",
        expected_model,
    )
    df = _read_csv_from_bundle(bundle, "patient_metrics.csv").copy()

    if len(df) != 20 or df["patient_id"].nunique() != 20:
        raise RuntimeError(
            f"{expected_model}: expected 20 unique locked-test patients, "
            f"found rows={len(df)}, unique={df['patient_id'].nunique()}."
        )

    df["patient_id"] = df["patient_id"].astype(int)
    df = df.sort_values("patient_id").reset_index(drop=True)
    return bundle, df


def assert_same_patients(reference_df, comparison_df, label):
    ref_ids = reference_df["patient_id"].astype(int).tolist()
    cmp_ids = comparison_df["patient_id"].astype(int).tolist()
    if ref_ids != cmp_ids:
        raise RuntimeError(
            f"Patient IDs/order do not match for {label}."
        )


def assert_same_oof_folds(reference_df, comparison_df, label):
    merged = reference_df[["patient_id", "fold"]].merge(
        comparison_df[["patient_id", "fold"]],
        on="patient_id",
        suffixes=("_reference", "_comparison"),
        validate="one_to_one",
    )
    if not np.array_equal(
        merged["fold_reference"].to_numpy(),
        merged["fold_comparison"].to_numpy(),
    ):
        raise RuntimeError(f"OOF fold assignments differ for {label}.")


In [3]:

OOF_MODELS = {
    "VentiMorph OOF": "VentiMorph-RelNet-v2.8.1-segmentation-first",
    "FCN-8s OOF": "FCN-8s",
    "SegNet OOF": "SegNet",
    "U-Net OOF": "U-Net",
    "U-Net++ OOF": "Reimplemented U-Net++ baseline under common protocol",
    "A1 OOF": "A1 — No Relation Refinement",
    "A2 OOF": "A2 — No Spatial Modality Gate",
    "A3 OOF": "A3 — No Deep Supervision",
    "A4 OOF": "A4 — No Stop-Gradient Guidance",
}

TEST_MODELS = {
    "VentiMorph test": "VentiMorph-RelNet-v2.8.1-segmentation-first",
    "FCN-8s test": "Reimplemented FCN-8s baseline under common protocol",
    "SegNet test": "Reimplemented SegNet baseline under common protocol",
    "U-Net test": "Reimplemented U-Net baseline under common protocol",
    "U-Net++ test": "Reimplemented U-Net++ baseline under common protocol",
}

audit_rows = []

oof_loaded = {}
for label, model_id in OOF_MODELS.items():
    bundle, df = load_oof_model(model_id)
    oof_loaded[label] = (bundle, df)

    finite_primary = np.isfinite(
        pd.to_numeric(df["mean_foreground_dice"], errors="coerce")
    ).all()

    audit_rows.append({
        "result": label,
        "stage": "OOF",
        "source": str(bundle["source"]),
        "metadata_model": bundle["metadata"].get("model"),
        "split_hash": bundle["metadata"].get("split_sha256"),
        "split_hash_ok": bundle["metadata"].get("split_sha256") == EXPECTED_SPLIT_SHA256,
        "rows": len(df),
        "unique_patients": df["patient_id"].nunique(),
        "expected_patient_count_ok": len(df) == 80 and df["patient_id"].nunique() == 80,
        "locked_test_evaluated_metadata": bundle["metadata"].get("locked_test_evaluated"),
        "primary_metric_all_finite": bool(finite_primary),
    })

test_loaded = {}
for label, model_id in TEST_MODELS.items():
    bundle, df = load_test_model(model_id)
    test_loaded[label] = (bundle, df)

    finite_primary = np.isfinite(
        pd.to_numeric(df["mean_foreground_dice"], errors="coerce")
    ).all()

    audit_rows.append({
        "result": label,
        "stage": "locked_test",
        "source": str(bundle["source"]),
        "metadata_model": bundle["metadata"].get("model"),
        "split_hash": bundle["metadata"].get("split_sha256"),
        "split_hash_ok": bundle["metadata"].get("split_sha256") == EXPECTED_SPLIT_SHA256,
        "rows": len(df),
        "unique_patients": df["patient_id"].nunique(),
        "expected_patient_count_ok": len(df) == 20 and df["patient_id"].nunique() == 20,
        "locked_test_evaluated_metadata": True,
        "primary_metric_all_finite": bool(finite_primary),
    })

# OOF fold alignment against VentiMorph.
reference_oof = oof_loaded["VentiMorph OOF"][1][["patient_id", "fold"]].copy()
for label, (bundle, df) in oof_loaded.items():
    assert_same_patients(
        oof_loaded["VentiMorph OOF"][1],
        df,
        label,
    )
    assert_same_oof_folds(
        oof_loaded["VentiMorph OOF"][1],
        df,
        label,
    )

# Locked-test patient alignment.
for label, (bundle, df) in test_loaded.items():
    assert_same_patients(
        test_loaded["VentiMorph test"][1],
        df,
        label,
    )

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

if not audit_df["split_hash_ok"].all():
    raise RuntimeError("At least one result has the wrong frozen split hash.")

if not audit_df["expected_patient_count_ok"].all():
    raise RuntimeError("At least one result has the wrong patient count.")

if not audit_df["primary_metric_all_finite"].all():
    raise RuntimeError("At least one result has non-finite Mean FG Dice.")

# Explicitly enforce no locked-test evaluation in OOF/ablation metadata.
oof_meta = audit_df[audit_df["stage"] == "OOF"]
if not (oof_meta["locked_test_evaluated_metadata"] == False).all():
    raise RuntimeError(
        "At least one OOF result metadata does not explicitly protect the locked test."
    )

print("\nAUDIT PASSED")
print("- All OOF results: 80 unique patients")
print("- All OOF fold assignments: identical")
print("- All locked-test results: 20 identical patients")
print("- Frozen split hash: identical")
print("- All primary Mean FG Dice values: finite")
print("- OOF/ablations: locked test not evaluated")


,result,stage,source,metadata_model,split_hash,split_hash_ok,rows,unique_patients,expected_patient_count_ok,locked_test_evaluated_metadata,primary_metric_all_finite
0,VentiMorph OOF,OOF,/kaggle/input/datasets/saifsamiul/ms3seg-final...,VentiMorph-RelNet-v2.8.1-segmentation-first,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,80,80,True,False,True
1,FCN-8s OOF,OOF,/kaggle/input/datasets/saifsamiul/ms3seg-final...,FCN-8s,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,80,80,True,False,True
2,SegNet OOF,OOF,/kaggle/input/datasets/saifsamiul/ms3seg-final...,SegNet,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,80,80,True,False,True
3,U-Net OOF,OOF,/kaggle/input/datasets/saifsamiul/ms3seg-final...,U-Net,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,80,80,True,False,True
4,U-Net++ OOF,OOF,/kaggle/input/datasets/saifsamiul/ms3seg-final...,Reimplemented U-Net++ baseline under common pr...,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,80,80,True,False,True
5,A1 OOF,OOF,/kaggle/input/datasets/saifsamiul/ms3seg-final...,A1 — No Relation Refinement,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,80,80,True,False,True
6,A2 OOF,OOF,/kaggle/input/datasets/saifsamiul/ms3seg-final...,A2 — No Spatial Modality Gate,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,80,80,True,False,True
7,A3 OOF,OOF,/kaggle/input/datasets/saifsamiul/ms3seg-final...,A3 — No Deep Supervision,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,80,80,True,False,True
8,A4 OOF,OOF,/kaggle/input/datasets/saifsamiul/ms3seg-final...,A4 — No Stop-Gradient Guidance,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,80,80,True,False,True
9,VentiMorph test,locked_test,/kaggle/input/datasets/saifsamiul/ms3seg-final...,VentiMorph-RelNet-v2.8.1-segmentation-first,d5b26261c93305d0554dae92454cee9363c8df5d946c94...,True,20,20,True,True,True



AUDIT PASSED
- All OOF results: 80 unique patients
- All OOF fold assignments: identical
- All locked-test results: 20 identical patients
- Frozen split hash: identical
- All primary Mean FG Dice values: finite
- OOF/ablations: locked test not evaluated


In [4]:

OUTPUT_DIR = WORK_ROOT / "reproducibility_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

audit_df.to_csv(
    OUTPUT_DIR / "reproducibility_audit.csv",
    index=False,
)

report = f"""MS3SEG / VentiMorph Reproducibility Audit

Frozen split SHA256:
{EXPECTED_SPLIT_SHA256}

OOF bundles audited: {sum(audit_df['stage'] == 'OOF')}
Locked-test bundles audited: {sum(audit_df['stage'] == 'locked_test')}

AUDIT PASSED:
- all OOF bundles contain 80 unique patients;
- all OOF fold assignments match the VentiMorph reference;
- all locked-test bundles contain the same 20 patients;
- all available split hashes match the frozen split;
- OOF metadata reports locked_test_evaluated=False;
- Mean foreground Dice is finite for every audited patient.
"""

(OUTPUT_DIR / "reproducibility_audit_report.txt").write_text(
    report,
    encoding="utf-8",
)

archive = shutil.make_archive(
    "/kaggle/working/MS3SEG_REPRODUCIBILITY_AUDIT_RESULTS",
    "zip",
    root_dir=OUTPUT_DIR,
)
print("Download:", archive)


Download: /kaggle/working/MS3SEG_REPRODUCIBILITY_AUDIT_RESULTS.zip
